# HAM10000 — Clasificación comparativa de métodos de augmentación sintética

**Pregunta de investigación:** ¿puede la augmentación con imágenes sintéticas mejorar la detección de melanoma frente a un baseline entrenado solo con datos reales? ¿Qué método generativo produce la mayor mejora?

## Diseño experimental

| Escenario | Train melanoma | Método generativo | Pregunta central |
|---|---|---|---|
| `real_only` | ~801 reales | — | Baseline sin augmentación sintética |
| `real_2x_ti` | 801 real + 801 TI | Textual Inversion (SD v1.5) | ¿TI mejora el Recall de melanoma? |
| `real_2x_lora` | 801 real + 801 LoRA | LoRA fine-tuning (SD v1.5) | ¿LoRA supera a TI? |
| `real_2x_gan` | 801 real + 801 GAN | WGAN-GP 64×64 px | ¿Una GAN clásica es competitiva con SD? |
| `real_2x_derm` | 801 real + 801 Derm | Derm-T2IM img2img (s=0.40) | ¿Un modelo dermoscopy-specific reduce el distributional shift? |
| `synthetic_only_ti` | 801 TI (sin reales mel) | Textual Inversion | ¿Las sintéticas pueden sustituir a las reales? |

**Control de cantidad:** todos los escenarios `real_2x_*` usan exactamente `N_REAL_MEL` sintéticas,
el mismo volumen que el conjunto real de melanoma. Esto hace los resultados comparables entre generadores.

**Test set:** siempre 100% imágenes reales (nunca contaminado con sintéticas).

**Clasificador:** EfficientNet-B0 pretrained (ImageNet), 15 épocas, LR=1e-4, CosineAnnealingLR, WeightedRandomSampler, semilla=42.

> Diseño inspirado en: Akrout et al. (2023) *Diffusion-based Data Augmentation for Skin Disease Classification*. arXiv:2301.04802


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Entorno:", "Google Colab" if IN_COLAB else "Local")


In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import timm
except ImportError:
    pip("timm")
    import timm

try:
    import sklearn
except ImportError:
    pip("scikit-learn")

try:
    from tqdm.auto import tqdm
except ImportError:
    pip("tqdm")
    from tqdm.auto import tqdm

print("Dependencias OK")


In [ ]:
import shutil, zipfile
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')

    # ── Estructura esperada en Drive ────────────────────────────────────────
    # MyDrive/ham10000-augmentation/
    #   data/
    #     classification_data.zip        ← imágenes reales (procesadas + splits)
    #     melanoma_train_for_colab.zip
    #     synthetic/
    #       textual_inversion/           ← imágenes TI (*.jpg)
    #       img2img/                     ← imágenes img2img (*.jpg)
    #       lora/                        ← imágenes LoRA (*.jpg)
    #       gan_final/                   ← imágenes GAN (*.png)
    #       derm_s040/                   ← imágenes Derm-T2IM s=0.40 (*.jpg)
    #       derm_s005/                   ← imágenes Derm-T2IM s=0.05 (*.jpg)
    #   models/                          ← embeddings TI
    #   experiments/                     ← resultados de runs
    # ────────────────────────────────────────────────────────────────────────

    DRIVE_ROOT = Path('/content/drive/MyDrive/ham10000-augmentation')
    # Buscar el ZIP en la ubicación nueva (data/) o antigua (raíz)
    _zip_new = DRIVE_ROOT / 'data' / 'classification_data.zip'
    _zip_old = DRIVE_ROOT / 'classification_data.zip'
    ZIP_PATH   = _zip_new if _zip_new.exists() else _zip_old
    IMAGES_DIR = Path('/content/images')
    SPLITS_DIR = Path('/content/splits')
    EXP_ROOT   = DRIVE_ROOT / 'experiments'

    # Descomprimir imágenes reales (solo la primera vez)
    if not IMAGES_DIR.exists():
        print("Descomprimiendo classification_data.zip ...")
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall('/content')
        print("Listo")

    # Copiar sintéticas a disco local (I/O más rápido que leer de Drive)
    LOCAL_SYNTH = Path('/content/synthetic')
    if not LOCAL_SYNTH.exists():
        print("Copiando synthetic/ de Drive a /content/synthetic/ ...")
        shutil.copytree(str(DRIVE_ROOT / 'data' / 'synthetic'), str(LOCAL_SYNTH))
        n = len(list(LOCAL_SYNTH.glob('**/*.jpg'))) + len(list(LOCAL_SYNTH.glob('**/*.png')))
        print(f"  {n} imágenes copiadas")

    SYNTH_ROOT = LOCAL_SYNTH

else:
    PROJECT_ROOT = Path.cwd()
    IMAGES_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'images'
    SPLITS_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'splits'
    SYNTH_ROOT   = PROJECT_ROOT / 'data' / 'synthetic'
    EXP_ROOT     = PROJECT_ROOT / 'experiments'

EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Subdirectorios de sintéticas por generador (alineados con la estructura Drive)
SYNTH_DIRS = {
    'ti':   SYNTH_ROOT / 'textual_inversion',
    'lora': SYNTH_ROOT / 'lora',
    'gan':  SYNTH_ROOT / 'gan_final',
    'derm': SYNTH_ROOT / 'derm_s040',
}

print("Paths configurados:")
print(f"  IMAGES_DIR : {IMAGES_DIR}  (existe: {IMAGES_DIR.exists()})")
print(f"  SPLITS_DIR : {SPLITS_DIR}  (existe: {SPLITS_DIR.exists()})")
print(f"  SYNTH_ROOT : {SYNTH_ROOT}  (existe: {SYNTH_ROOT.exists()})")
for k, d in SYNTH_DIRS.items():
    exts = list(d.glob('*.jpg')) + list(d.glob('*.png')) if d.exists() else []
    print(f"  synth/{k:4s} : {d.name:<22} ({len(exts)} imgs)")


## Estado de progreso

Ejecutar esta celda al reconectar para ver qué escenarios ya están completos sin necesidad de recargar el modelo.


In [ ]:
import json

SCENARIO_KEYS = [
    'real_only',
    'real_2x_ti',
    'real_2x_lora',
    'real_2x_gan',
    'real_2x_derm',
    'synthetic_only_ti',
]

def find_run_dir(scenario):
    completed = sorted([
        d for d in EXP_ROOT.glob(f'*_{scenario}')
        if (d / 'test_metrics.json').exists()
    ])
    return completed[-1] if completed else None

print(f"{'Escenario':<22} {'Estado':<12} {'AUC':>6} {'Recall':>7} {'F1 mel':>7}")
print("-" * 60)
for sc in SCENARIO_KEYS:
    run_dir = find_run_dir(sc)
    if run_dir:
        m = json.loads((run_dir / 'test_metrics.json').read_text())
        print(f"  {sc:<20} ✅ done     {m['auc']:>6.4f} {m['recall_mel']:>7.4f} {m['f1_mel']:>7.4f}")
    else:
        in_progress = sorted(EXP_ROOT.glob(f'*_{sc}'))
        if in_progress and (in_progress[-1] / 'checkpoint_last.pt').exists():
            print(f"  {sc:<20} 🔄 resume")
        else:
            print(f"  {sc:<20} ⬜ pendiente")


## Configuración de escenarios e hiperparámetros

In [ ]:
import pandas as pd
import torch

# ── Reproducibilidad ──
SEED = 42
torch.manual_seed(SEED)

# ── Hardware ──
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE  = torch.float32
    BATCH  = 32
    print(f"CUDA: {torch.cuda.get_device_name(0)}  "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE  = torch.float32
    BATCH  = 16
    print("MPS (Apple Silicon)")
else:
    DEVICE = torch.device('cpu')
    DTYPE  = torch.float32
    BATCH  = 8
    print("CPU — entrenamiento lento")

NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0

# ── Hiperparámetros (fijos en todos los escenarios) ──
EPOCHS = 15
LR     = 1e-4

# ── Conteo de reales para fijar N_SYNTH en escenarios 2x ──
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
N_REAL_MEL = int((train_df['dx'] == 'mel').sum())
print(f"\nMelanoma reales en train: {N_REAL_MEL}")

# ── Definición de escenarios ──────────────────────────────────────────────────
# Cada escenario define exactamente qué entra en el train set de melanoma.
# 'synth_key' apunta a SYNTH_DIRS; 'synth_n' controla cuántas sintéticas se usan.
# 'mel_real=False' reemplaza los reales por sintéticas (synthetic_only).
SCENARIOS = {
    'real_only': {
        'label':       'Real only (baseline)',
        'description': 'Solo imágenes reales de melanoma. Establece el rendimiento base.',
        'mel_real':    True,
        'synth_key':   None,
        'synth_n':     0,
        'reference':   '—',
    },
    'real_2x_ti': {
        'label':       'Real + Textual Inversion (2×)',
        'description': 'Augmentación con token <mel-skin> entrenado 5000 steps sobre SD v1.5.',
        'mel_real':    True,
        'synth_key':   'ti',
        'synth_n':     N_REAL_MEL,
        'reference':   'Akrout et al. 2023',
    },
    'real_2x_lora': {
        'label':       'Real + LoRA (2×)',
        'description': 'LoRA fine-tuning (rank=32) sobre SD v1.5 entrenado en melanoma.',
        'mel_real':    True,
        'synth_key':   'lora',
        'synth_n':     N_REAL_MEL,
        'reference':   '—',
    },
    'real_2x_gan': {
        'label':       'Real + WGAN-GP (2×)',
        'description': 'Generador WGAN-GP entrenado 100 epochs, imágenes 64×64 px redimensionadas a 224.',
        'mel_real':    True,
        'synth_key':   'gan',
        'synth_n':     N_REAL_MEL,
        'reference':   '—',
    },
    'real_2x_derm': {
        'label':       'Real + Derm-T2IM (2×)',
        'description': 'img2img con modelo dermoscopy-specific (strength=0.40). Menor distributional shift esperado.',
        'mel_real':    True,
        'synth_key':   'derm',
        'synth_n':     N_REAL_MEL,
        'reference':   'Hasan et al. 2024 (Derm-T2IM)',
    },
    'synthetic_only_ti': {
        'label':       'Synthetic only — TI',
        'description': 'Sin imágenes reales de melanoma en train. Evalúa si las sintéticas pueden sustituir a las reales.',
        'mel_real':    False,
        'synth_key':   'ti',
        'synth_n':     N_REAL_MEL,
        'reference':   'Akrout et al. 2023',
    },
}

print(f"\n{'Escenario':<22} {'Mel real':>9} {'Mel synth':>10} {'Generador'}")
print("-" * 60)
for sc, cfg in SCENARIOS.items():
    n_real  = N_REAL_MEL if cfg['mel_real'] else 0
    n_synth = cfg['synth_n']
    gen     = cfg['synth_key'] or '—'
    print(f"  {sc:<20} {n_real:>9} {n_synth:>10}   {gen}")


## Dataset y DataLoaders

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# Fija semillas para reproducibilidad en shuffle y sampling
random.seed(SEED)
np.random.seed(SEED)

TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

EVAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

CLASS_MAP = {'nv': 0, 'mel': 1}


class SkinDataset(Dataset):
    '''Dataset que combina imágenes reales (referenciadas por CSV) con sintéticas (carpeta).'''

    def __init__(self, records, transform):
        # records: list of (path_str, label_int)
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label = self.records[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        return self.transform(img), label


def build_records(split_csv, images_dir, scenario_cfg, synth_dirs, seed=42):
    '''Construye la lista de (path, label) para un escenario.
    Nv siempre viene del split real. Mel puede ser real, sintético o ambos.
    '''
    df = pd.read_csv(split_csv)
    rng = random.Random(seed)

    # Nv — siempre real
    def _resolve(img_id):
        p = images_dir / (img_id + '.jpg')
        if not p.exists():
            p = images_dir / img_id  # por si ya trae extensión
        return p
    nv_records = [
        (str(_resolve(row['image_id'])), CLASS_MAP['nv'])
        for _, row in df[df['dx'] == 'nv'].iterrows()
        if _resolve(row['image_id']).exists()
    ]

    # Mel real
    mel_real = []
    if scenario_cfg['mel_real']:
        mel_real = [
            (str(_resolve(row['image_id'])), CLASS_MAP['mel'])
            for _, row in df[df['dx'] == 'mel'].iterrows()
            if _resolve(row['image_id']).exists()
        ]

    # Mel sintético
    mel_synth = []
    if scenario_cfg['synth_key'] and scenario_cfg['synth_n'] > 0:
        synth_dir = synth_dirs[scenario_cfg['synth_key']]
        # Acepta .jpg y .png
        candidates = list(synth_dir.glob('*.jpg')) + list(synth_dir.glob('*.png'))
        if len(candidates) < scenario_cfg['synth_n']:
            print(f"  ⚠ Solo {len(candidates)} imágenes en {synth_dir.name} "
                  f"(se pedían {scenario_cfg['synth_n']})")
        chosen = rng.sample(candidates, min(scenario_cfg['synth_n'], len(candidates)))
        mel_synth = [(str(p), CLASS_MAP['mel']) for p in chosen]

    all_records = nv_records + mel_real + mel_synth
    rng.shuffle(all_records)
    return all_records


def make_loaders(scenario_cfg, synth_dirs, splits_dir, images_dir, batch, num_workers):
    train_records = build_records(
        splits_dir / 'train.csv', images_dir, scenario_cfg, synth_dirs
    )
    val_records = build_records(
        splits_dir / 'val.csv', images_dir,
        {**scenario_cfg, 'synth_key': None, 'synth_n': 0, 'mel_real': True},
        synth_dirs
    )
    test_records = build_records(
        splits_dir / 'test.csv', images_dir,
        {**scenario_cfg, 'synth_key': None, 'synth_n': 0, 'mel_real': True},
        synth_dirs
    )

    # WeightedRandomSampler para compensar desbalance residual en train
    labels  = [r[1] for r in train_records]
    counts  = [labels.count(c) for c in [0, 1]]
    weights = [1.0 / counts[l] for l in labels]
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                    generator=torch.Generator().manual_seed(SEED))

    train_loader = DataLoader(SkinDataset(train_records, TRAIN_TF), batch_size=batch,
                              sampler=sampler, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(SkinDataset(val_records, EVAL_TF),   batch_size=batch,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(SkinDataset(test_records, EVAL_TF),  batch_size=batch,
                              shuffle=False, num_workers=num_workers, pin_memory=True)

    print(f"  train: {len(train_records)} ({labels.count(1)} mel / {labels.count(0)} nv)  "
          f"val: {len(val_records)}  test: {len(test_records)}")
    return train_loader, val_loader, test_loader

print("Dataset utilities listas")


## Modelo y utilidades de entrenamiento

In [ ]:
import timm
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score, roc_auc_score, recall_score, precision_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# Compatibilidad torch >= 2.6 (weights_only=True rompe timm/numpy al cargar checkpoints)
try:
    import numpy as _np
    torch.serialization.add_safe_globals([_np._core.multiarray.scalar])
except (AttributeError, ImportError):
    pass


def build_model():
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)
    model = model.to(DEVICE)
    return model


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)
    return {
        'auc':           float(roc_auc_score(all_labels, all_probs)),
        'recall_mel':    float(recall_score(all_labels, preds, pos_label=1, zero_division=0)),
        'precision_mel': float(precision_score(all_labels, preds, pos_label=1, zero_division=0)),
        'f1_mel':        float(f1_score(all_labels, preds, pos_label=1, zero_division=0)),
        'f1_nv':         float(f1_score(all_labels, preds, pos_label=0, zero_division=0)),
        'accuracy':      float(np.mean(np.array(all_labels) == preds)),
        '_labels':       all_labels,
        '_probs':        all_probs,
    }


def save_plots(run_dir, history, test_metrics):
    # Curvas de entrenamiento
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], label='train'); axes[0].plot(epochs, history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Época')
    axes[1].plot(epochs, history['val_auc'], label='AUC'); axes[1].plot(epochs, history['val_f1_mel'], label='F1 mel')
    axes[1].set_title('Validación'); axes[1].legend(); axes[1].set_xlabel('Época')
    fig.savefig(run_dir / 'training_curves.png', dpi=100, bbox_inches='tight')
    plt.close(fig)

    # Confusion matrix
    labels, probs = test_metrics.pop('_labels'), test_metrics.pop('_probs')
    preds = (np.array(probs) >= 0.5).astype(int)
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['nv', 'mel']).plot(ax=ax)
    ax.set_title('Test — Confusion Matrix')
    fig.savefig(run_dir / 'confusion_matrix.png', dpi=100, bbox_inches='tight')
    plt.close(fig)

    # ROC curve
    fpr, tpr, _ = roc_curve(labels, probs)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC={test_metrics['auc']:.3f}")
    ax.plot([0,1],[0,1],'--', color='gray')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC — Test')
    ax.legend(); fig.savefig(run_dir / 'roc_curve.png', dpi=100, bbox_inches='tight')
    plt.close(fig)


print("Model utilities listas")


## Función run_scenario — entrena un escenario con resume

In [ ]:
def run_scenario(scenario_name, scenario_cfg):
    '''Entrena EfficientNet-B0 para un escenario.
    - Omite si test_metrics.json ya existe (completion marker).
    - Retoma desde checkpoint_last.pt si existe y el escenario está incompleto.
    '''
    print(f"\n{'='*60}")
    print(f"Escenario: {scenario_name}")
    print(f"  {scenario_cfg['label']}")
    print(f"  {scenario_cfg['description']}")

    # ── Completion check ──────────────────────────────────────────────────────
    existing = find_run_dir(scenario_name)
    if existing:
        m = json.loads((existing / 'test_metrics.json').read_text())
        print(f"  ✅ Ya completado: {existing.name}")
        print(f"     AUC={m['auc']:.4f}  Recall={m['recall_mel']:.4f}  F1={m['f1_mel']:.4f}")
        return m

    # ── Nuevo run ─────────────────────────────────────────────────────────────
    ts      = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    run_dir = EXP_ROOT / f'{ts}_{scenario_name}'
    run_dir.mkdir(parents=True, exist_ok=True)

    # ── DataLoaders ───────────────────────────────────────────────────────────
    print("  Construyendo dataloaders ...")
    train_loader, val_loader, test_loader = make_loaders(
        scenario_cfg, SYNTH_DIRS, SPLITS_DIR, IMAGES_DIR, BATCH, NUM_WORKERS
    )

    # ── Modelo, optimizador, scheduler ───────────────────────────────────────
    model     = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history    = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1_mel': []}
    start_epoch = 0
    best_f1     = 0.0
    ckpt_path   = run_dir / 'checkpoint_last.pt'

    # ── Resume desde checkpoint ───────────────────────────────────────────────
    if ckpt_path.exists():
        print("  🔄 Retomando desde checkpoint_last.pt ...")
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        history     = ckpt['history']
        start_epoch = ckpt['epoch'] + 1
        best_f1     = ckpt['best_f1']
        print(f"  Retomando desde época {start_epoch}")

    # ── Loop de entrenamiento ─────────────────────────────────────────────────
    best_model_path = run_dir / 'best_model.pt'

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"  Época {epoch+1}/{EPOCHS}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()
        train_loss /= len(train_loader)

        # Validación
        val_metrics = evaluate(model, val_loader)
        val_metrics.pop('_labels', None); val_metrics.pop('_probs', None)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(0.0)   # CrossEntropy en val omitida para velocidad
        history['val_auc'].append(val_metrics['auc'])
        history['val_f1_mel'].append(val_metrics['f1_mel'])

        print(f"  E{epoch+1:02d}  loss={train_loss:.4f}  "
              f"AUC={val_metrics['auc']:.4f}  "
              f"Recall={val_metrics['recall_mel']:.4f}  "
              f"F1={val_metrics['f1_mel']:.4f}")

        # Guardar mejor modelo por F1 mel
        if val_metrics['f1_mel'] >= best_f1:
            best_f1 = val_metrics['f1_mel']
            torch.save(model.state_dict(), best_model_path)

        # Checkpoint de resume (sobrescribe cada época)
        torch.save({
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'history':   history,
            'epoch':     epoch,
            'best_f1':   best_f1,
        }, ckpt_path)

    # ── Evaluación en test (carga mejor modelo) ───────────────────────────────
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    test_metrics = evaluate(model, test_loader)

    print(f"  TEST → AUC={test_metrics['auc']:.4f}  "
          f"Recall={test_metrics['recall_mel']:.4f}  "
          f"F1={test_metrics['f1_mel']:.4f}")

    # ── Guardar artefactos ────────────────────────────────────────────────────
    (run_dir / 'history.json').write_text(json.dumps(history, indent=2))
    save_plots(run_dir, history, test_metrics)

    config = {
        'run_id':          run_dir.name,
        'scenario':        scenario_name,
        'label':           scenario_cfg['label'],
        'description':     scenario_cfg['description'],
        'reference':       scenario_cfg['reference'],
        'model':           'efficientnet_b0',
        'pretrained':      True,
        'epochs':          EPOCHS,
        'lr':              LR,
        'weight_decay':    1e-4,
        'batch_size':      BATCH,
        'seed':            SEED,
        'n_real_mel':      N_REAL_MEL if scenario_cfg['mel_real'] else 0,
        'n_synth_mel':     scenario_cfg['synth_n'],
        'synth_source':    scenario_cfg['synth_key'],
        'dataset': {
            'source':       'Harvard Dataverse',
            'doi':          'doi:10.7910/DVN/DBW86T',
            'version':      4,
            'release_date': '2023-02-07',
        },
        'started_at':  ts,
        'test_metrics': {k:v for k,v in test_metrics.items() if not k.startswith('_')},
    }
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2))

    # Completion marker — se escribe al final para que el resume funcione
    clean_metrics = {k: v for k, v in test_metrics.items() if not k.startswith('_')}
    (run_dir / 'test_metrics.json').write_text(json.dumps(clean_metrics, indent=2))

    # Borrar checkpoint una vez completado
    ckpt_path.unlink(missing_ok=True)

    print(f"  Guardado en {run_dir.name}")
    return clean_metrics


print("run_scenario lista")


## Ejecución de los 6 escenarios

Ejecutar esta celda corre todos los escenarios pendientes en orden.
Los escenarios ya completados se saltan automáticamente.
Si la sesión se interrumpe, al volver a ejecutar retoma desde el último checkpoint.


In [ ]:
all_results = {}

for sc_name, sc_cfg in SCENARIOS.items():
    metrics = run_scenario(sc_name, sc_cfg)
    all_results[sc_name] = metrics

print("\n✅ Todos los escenarios completados")


## Resultados comparativos

In [ ]:
# Recolectar resultados (incluye runs de sesiones anteriores)
results_rows = []
for sc_name, sc_cfg in SCENARIOS.items():
    run_dir = find_run_dir(sc_name)
    if run_dir:
        m   = json.loads((run_dir / 'test_metrics.json').read_text())
        cfg = json.loads((run_dir / 'config.json').read_text()) if (run_dir/'config.json').exists() else {}
        results_rows.append({
            'Escenario':    sc_name,
            'Generador':    sc_cfg['synth_key'] or '—',
            'Mel real':     cfg.get('n_real_mel', '?'),
            'Mel synth':    cfg.get('n_synth_mel', '?'),
            'AUC':          m['auc'],
            'Recall mel':   m['recall_mel'],
            'Prec. mel':    m['precision_mel'],
            'F1 mel':       m['f1_mel'],
            'Accuracy':     m['accuracy'],
        })
    else:
        results_rows.append({'Escenario': sc_name, 'Generador': '—', 'AUC': '—'})

results_df = pd.DataFrame(results_rows)
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

# Guardar CSV de resultados consolidados
results_df.to_csv(EXP_ROOT / 'comparative_results.csv', index=False)
print(f"\nGuardado: {EXP_ROOT}/comparative_results.csv")


In [ ]:
# Gráfico comparativo de métricas clave
numeric_df = results_df[results_df['AUC'] != '—'].copy()
numeric_df[['AUC','Recall mel','F1 mel']] = numeric_df[['AUC','Recall mel','F1 mel']].astype(float)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_to_plot = [('AUC', 'AUC-ROC'), ('Recall mel', 'Recall melanoma'), ('F1 mel', 'F1 melanoma')]

for ax, (col, title) in zip(axes, metrics_to_plot):
    bars = ax.bar(range(len(numeric_df)), numeric_df[col], color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(numeric_df)))
    ax.set_xticklabels(numeric_df['Escenario'], rotation=30, ha='right', fontsize=8)
    ax.set_title(title); ax.set_ylim(0, 1)
    # Línea del baseline
    baseline = numeric_df[numeric_df['Escenario']=='real_only'][col].values
    if len(baseline):
        ax.axhline(baseline[0], color='red', linestyle='--', linewidth=1, label='baseline')
        ax.legend(fontsize=8)
    for bar, val in zip(bars, numeric_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('Comparación de métodos de augmentación sintética\n(EfficientNet-B0, test set real)',
             fontsize=12)
plt.tight_layout()
plt.savefig(EXP_ROOT / 'comparative_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico guardado en comparative_results.png")
